In [0]:
meu_schema = "workspace.projeto_analytics_2"

In [0]:
df_aeronave_raw = spark.read.table(f"{meu_schema}.aeronautica_aeronave")

df_aeronautica_raw = spark.read.table(f"{meu_schema}.aeronautica_fator")

df_ocorrencia_raw = spark.read.table(f"{meu_schema}.aeronautica_ocorrencia")

In [0]:
from pyspark.sql import functions as f

valores_sujos = ["***", "NULL"]

df_aeronave_silver = df_aeronave_raw.withColumn("aeronave_motor_tipo", f.when(f.col("aeronave_motor_tipo").isin(valores_sujos), f.lit("NAO INFORMADO")).otherwise(f.col("aeronave_motor_tipo")))

#display(df_aeronave_silver.limit(20))

In [0]:
from pyspark.sql import functions as f

df_aeronave_silver = df_aeronave_silver.withColumn(
    "aeronave_assentos", 
    f.when(f.col("aeronave_assentos") == "NULL", f.lit(None)).otherwise(f.col("aeronave_assentos"))
)

#display(df_aeronave_silver.limit(20))

In [0]:
df_aeronave_silver = df_aeronave_silver.withColumn(
    "aeronave_assentos", 
    f.col("aeronave_assentos").cast("int"))

#display(df_aeronave_silver.limit(20))

In [0]:
df_aeronave_silver = df_aeronave_silver.withColumn("aeronave_assentos", f.when(f.col("aeronave_assentos"). isNull(), f.lit(0)).otherwise(f.col("aeronave_assentos")))

#display(df_aeronave_silver.limit(20))

In [0]:
df_aeronave_silver = df_aeronave_silver.withColumn("aeronave_ano_fabricacao", f.when((f.col("aeronave_ano_fabricacao") == "NULL") | (f.col("aeronave_ano_fabricacao") =="0"), f.lit("NAO INFORMADO")).otherwise(f.col("aeronave_ano_fabricacao")))
#display(df_aeronave_silver.limit(20))

In [0]:
Valores = ["NULL", "***", " "]
Colunas = ['aeronave_matricula',
 'aeronave_operador_categoria',
 'aeronave_tipo_veiculo',
 'aeronave_fabricante',
 'aeronave_modelo',
 'aeronave_tipo_icao', 'aeronave_motor_tipo',
 'aeronave_motor_quantidade', 'aeronave_ano_fabricacao',
 'aeronave_pais_fabricante',
 'aeronave_pais_registro',
 'aeronave_registro_categoria',
 'aeronave_registro_segmento',
 'aeronave_voo_origem',
 'aeronave_voo_destino',
 'aeronave_fase_operacao',
 'aeronave_tipo_operacao',
 'aeronave_nivel_dano']

for c in Colunas:
  df_aeronave_silver = df_aeronave_silver.withColumn(c, f.when(f.col(c).isin(Valores), f.lit(None)).otherwise(f.col(c)))

df_aeronave_silver = df_aeronave_silver.fillna("NAO INFORMADO")

#display(df_aeronave_silver.limit(20))

In [0]:
df_aeronave_silver = df_aeronave_silver.withColumn(
    "aeronave_pmd_categoria",
    f.when(f.col("aeronave_pmd") <= 500, f.lit("ULTRA LEVE"))
     .when(f.col("aeronave_pmd") <= 1000, f.lit("LEVE"))
     .when(f.col("aeronave_pmd") <= 5000, f.lit("MEDIO"))
     .when(f.col("aeronave_pmd") <= 20000, f.lit("PESADO"))
     .when(f.col("aeronave_pmd") > 20000, f.lit("MUITO PESADO"))
     .otherwise(f.lit("NAO INFORMADO"))
)

#display(df_aeronave_silver.limit(30))

In [0]:
df_aeronave_silver.write.format("delta").mode("overwrite").saveAsTable(f"{meu_schema}.aeronautica_aeronave_silver")